# 指数曲線から求めた個人別学習率の条件間比較

core MTと1/TPeについて、practice 1〜400試行の指数曲線の係数 **c（試行⁻¹）** を比較する。漸近値までの残差の大きさは1試行進むと exp(−c) 倍になる。表には **1試行当たりの残差縮小率 = 100 × (1−exp(−c)) %** も示す。この割合の分母は、その時点で漸近値までに残っている差である。

- core MT は a + b exp(−cN)、1/TPe は Y∞ + (Y₁−Y∞) exp(−c(N−1))。1/TPeは試行ごとに逆数を取る。個人別回帰でべき乗式を選んだ人も含め、30名全員に同じ指数モデルを適用する。
- 有効な試行番号ごとに参加者内で平均する。条件曲線は、同じ試行番号に有効値がある参加者を等重みで平均してから別に回帰する。検定に使うのは各人の係数cで、各指標につき独立な30観測（各群10名）。
- 基本採用条件は countedAsTrial=true かつ coreExitMs が有限。休憩境界周辺の除外、MT外れ値除外、successによる選別は行わない。前回と同じ実装・制約・3組の初期値を用い、数値計算は1コア。
- 全体比較は順位に基づくKruskal–Wallis検定（同順位補正、χ²近似、2指標でHolm補正）。各条件対は探索的に両側Mann–Whitney検定を行い、各組の20名を10名ずつに分ける全184,756通りから同順位に対応した正確なp値を求める。3条件対 × 2指標の6比較で別にHolm補正する。いずれも5%水準。各人の推定値を入力とする二段階の探索的比較で、曲線係数自体の推定誤差は検定に伝播しない。

In [1]:
import csv
import math
import os
import sys
from collections import Counter, defaultdict
from html import escape
from itertools import combinations
from pathlib import Path
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'analysis.py').is_file() and (p/'inputCSV').is_dir())
base = root/'output/learning_report'
out = base/'individual_exponential_rates_20261128_212205'
for name in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS','VECLIB_MAXIMUM_THREADS'):
    os.environ[name] = '1'
sys.path.insert(0,str(root))
import numpy as np
from scipy import stats
from IPython.display import HTML, display
from analysis import load_rows, trial_observations, fit_regression_model
conditions = ('C1','C2','C3')
models = {'core MT':'exponential','1/TPe':'inverse_TPe_exponential'}
metrics = list(models)

def save_csv(name, rows):
    with (out/name).open('w',encoding='utf-8-sig',newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def table(headers, rows):
    head = ''.join('<th>'+escape(str(x))+'</th>' for x in headers)
    body = ''.join('<tr>'+''.join('<td>'+escape(str(x))+'</td>' for x in row)+'</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

def shrink(c):
    return -100.*np.expm1(-c)

def test_value(c):
    for bound in (0.000001,5.):
        if math.isclose(c,bound,rel_tol=0.,abs_tol=1e-10):
            return bound
    return c

def fit_rate(members, metric):
    inv = metric == '1/TPe'
    x,y = trial_observations(members,'coreMtMs','TPe' if inv else 'MT',inverse=inv)
    fit = fit_regression_model(x,y,models[metric])
    if not fit['ok']:
        raise ValueError((metric,tuple(members),fit['message']))
    p = fit['parameters']
    c = p['c']
    first_gap = p['Y1']-p['Yinf'] if inv else p['b']*math.exp(-c)
    return {'有効試行番号数':len(x),'c':c,'1試行の残差縮小率_pct':float(shrink(c)),
            '試行1の予測値と漸近値の差':first_gap, 'R²':fit['r2'], 'RMSE':fit['rmse'],
            '推定警告':fit['warning']}

members = {c:defaultdict(list) for c in conditions}
assigned = defaultdict(set)
for r in load_rows(root/'inputCSV'):
    if r['phase']!='practice' or r['condition'] not in conditions or not 1<=r['n']<=400:
        continue
    assigned[r['participant']].add(r['condition'])
    if r['countedAsTrial'] and r['coreExitMs'] is not None:
        members[r['condition']][r['participant']].append(r)
assert len(assigned)==30 and all(len(s)==1 for s in assigned.values())
assert all(len(v)==10 for v in members.values())
people, summary = [], []
rates = {m:{} for m in metrics}
for c, group in members.items():
    for pid,rows in sorted(group.items()):
        person = {'条件':c,'参加者ID':pid}
        for m in metrics:
            rec = fit_rate({pid:rows},m)
            person.update({m+' '+k:v for k,v in rec.items()})
        people.append(person)
    for m in metrics:
        group_fit = fit_rate(group,m)
        cs = np.array([p[m+' c'] for p in people if p['条件']==c])
        rates[m][c] = np.array([test_value(v) for v in cs])
        q1,med,q3 = np.quantile(cs,[.25,.5,.75])
        summary.append({'条件':c,'指標':m,'人数':len(cs),'条件曲線c':group_fit['c'],
                        '条件曲線縮小率_pct':group_fit['1試行の残差縮小率_pct'],'条件曲線R²':group_fit['R²'],
                        '個人cの算術平均':cs.mean(),'個人cの中央値':med,'個人cの第1四分位':q1,'個人cの第3四分位':q3,
                        '個人縮小率の中央値_pct':np.median(shrink(cs)),'条件曲線の推定警告':group_fit['推定警告']})
def rate_note(p, m):
    if m == 'core MT' and p['参加者ID'] == '014':
        return '振幅ほぼ0・cを解釈不能'
    if test_value(p[m+' c']) == 5.:
        return 'cが上限5'
    return p[m+' 推定警告']

for person in people:
    for m in metrics:
        person[m+' 解釈上の注記'] = rate_note(person,m)
save_csv('individual_rates.csv',people)
save_csv('condition_rates.csv',summary)

## 条件平均曲線の学習率と個人別学習率の分布

条件曲線から推定したcと、個人別cの中央値・四分位範囲を併記する。曲線を平均する操作と係数を平均する操作は一般に異なる。

In [2]:
table(['指標','条件','条件曲線c','残差縮小率 (%/試行)','個人cの中央値 [Q1, Q3]','個人cの算術平均'],
      [[r['指標'],r['条件'],f"{r['条件曲線c']:.6f}",f"{r['条件曲線縮小率_pct']:.3f}",
        f"{r['個人cの中央値']:.6f} [{r['個人cの第1四分位']:.6f}, {r['個人cの第3四分位']:.6f}]",
        f"{r['個人cの算術平均']:.6f}"] for r in sorted(summary,key=lambda r:(metrics.index(r['指標']),r['条件']))])

指標,条件,条件曲線c,残差縮小率 (%/試行),"個人cの中央値 [Q1, Q3]",個人cの算術平均
core MT,C1,0.017367,1.722,"0.019735 [0.011869, 0.115476]",0.073967
core MT,C2,0.008645,0.861,"0.010388 [0.001289, 0.017829]",0.034054
core MT,C3,0.010273,1.022,"0.011821 [0.002178, 0.023030]",0.380884
1/TPe,C1,0.022180,2.194,"0.024542 [0.009421, 0.124498]",0.120278
1/TPe,C2,0.006206,0.619,"0.010957 [0.004869, 0.153444]",0.576647
1/TPe,C3,0.010732,1.067,"0.016681 [0.003644, 0.086878]",0.535243


## 個人別の学習率

指数曲線がほぼ平坦だと、係数cの値が大きくても学習が速いという解釈はできない。core MTの014は振幅がほぼ0でR²も0。1/TPeの003・014はcの推定上限5に達した。元の推定値はCSVに保持し、順位検定では同じ制約値の10⁻¹⁰以内にある推定値を同順位とする。「境界」の警告にはc以外のパラメータが境界に達した場合もある。残差縮小率はcの単調変換なので同じ順位比較となる。

In [3]:
for m in metrics:
    table(['条件','ID',m+' c (試行⁻¹)','残差縮小率 (%/試行)','R²','注記'],
          [[p['条件'],p['参加者ID'],f"{p[m+' c']:.6f}",f"{p[m+' 1試行の残差縮小率_pct']:.3f}",
            f"{p[m+' R²']:.4f}",p[m+' 解釈上の注記']] for p in people])

条件,ID,core MT c (試行⁻¹),残差縮小率 (%/試行),R²,注記
C1,011,0.010897,1.084,0.2568,
C1,012,0.049119,4.793,0.3466,
C1,015,0.002040,0.204,0.3160,パラメータが制約境界に達しています
C1,022,0.022660,2.241,0.7345,
C1,024,0.014784,1.468,0.6841,
C1,025,0.137595,12.855,0.5987,
C1,027,0.185650,16.944,0.0665,
C1,028,0.296330,25.646,0.1600,
C1,029,0.016810,1.667,0.6308,
C1,030,0.003782,0.378,0.3016,


条件,ID,1/TPe c (試行⁻¹),残差縮小率 (%/試行),R²,注記
C1,011,0.008823,0.878,0.1191,
C1,012,0.001225,0.122,0.0508,パラメータが制約境界に達しています
C1,015,0.002722,0.272,0.1427,パラメータが制約境界に達しています
C1,022,0.039608,3.883,0.2618,
C1,024,0.029521,2.909,0.3114,
C1,025,0.152794,14.169,0.2474,
C1,027,0.282915,24.642,0.0375,
C1,028,0.654389,48.024,0.1767,
C1,029,0.019564,1.937,0.2975,
C1,030,0.011218,1.116,0.1101,


## 個人別学習率の条件間検定

Kruskal–Wallis検定とMann–Whitney検定は推定係数の順位を比較する。群の分布形状が異なる場合、中央値だけの差としては解釈できない。Uは左条件を基準とし、順位二系列相関は正なら左条件のcが大きい傾向を表す。

In [4]:
def correct(rows):
    last = 0.
    for i,row in enumerate(sorted(rows,key=lambda r:r['未補正p'])):
        last = max(last,min(1.,(len(rows)-i)*row['未補正p']))
        row['Holm補正p'] = last
        row['有意性_5%'] = '有意' if last < .05 else '非有意'
    return rows

def omnibus(groups):
    return stats.kruskal(*(groups[c] for c in conditions))

def test_row(m, label, contrast, stat, df, effect, p, family):
    return {'指標':m,'検定':label,'比較':contrast,'統計量':float(stat),'自由度':df,
            '順位二系列相関':effect,'未補正p':float(p),'Holm補正範囲':family}

partitions = np.array(list(combinations(range(20),10)),dtype=np.int8)
def test_pair(x,y):
    ranks = stats.rankdata(np.r_[x,y],method='average')
    n = len(x)
    center = n*ranks.mean()
    observed = ranks[:n].sum()
    null = ranks[partitions].sum(axis=1)
    p = np.mean(np.abs(null-center) >= abs(observed-center))
    u = observed-n*(n+1)/2
    return u,2*u/(len(x)*len(y))-1,float(p)

global_tests, pairs = [], []
for m,groups in rates.items():
    h,p = omnibus(groups)
    global_tests.append(test_row(m,'Kruskal–Wallis H','C1・C2・C3',h,2,None,p,'全体比較2件'))
    for a,b in combinations(conditions,2):
        u,eff,pair_p = test_pair(groups[a],groups[b])
        pairs.append(test_row(m,'Mann–Whitney U (正確両側)',a+'–'+b,u,None,eff,pair_p,'指標2×条件対3の6件'))
tests = correct(global_tests)+correct(pairs)
save_csv('tests.csv',tests)
table(['指標','比較','検定','統計量','未補正p','Holm p','判定'],
      [[r['指標'],r['比較'],r['検定'],f"{r['統計量']:.4f}",f"{r['未補正p']:.5f}",
        f"{r['Holm補正p']:.5f}",r['有意性_5%']] for r in tests])

指標,比較,検定,統計量,未補正p,Holm p,判定
core MT,C1・C2・C3,Kruskal–Wallis H,2.4232,0.29772,0.59543,非有意
1/TPe,C1・C2・C3,Kruskal–Wallis H,0.3852,0.82479,0.82479,非有意
core MT,C1–C2,Mann–Whitney U (正確両側),72.0000,0.10512,0.63073,非有意
core MT,C1–C3,Mann–Whitney U (正確両側),63.0000,0.35268,1.00000,非有意
core MT,C2–C3,Mann–Whitney U (正確両側),50.0000,1.00000,1.00000,非有意
1/TPe,C1–C2,Mann–Whitney U (正確両側),57.0000,0.63053,1.00000,非有意
1/TPe,C1–C3,Mann–Whitney U (正確両側),57.0000,0.63053,1.00000,非有意
1/TPe,C2–C3,Mann–Whitney U (正確両側),48.5000,0.92972,1.00000,非有意


## core MTの014の順位への感度

振幅がほぼ0の014はcをデータから決めにくい。全30名を維持し、他の29名の推定値を固定して、許容範囲内で014が取り得るすべての順位（同順位も含む）に置いた。各順位で3群比較を再計算し、未補正p値の範囲を確認する。この範囲は信頼区間ではない。

In [5]:
group = rates['core MT']
pos = [p['参加者ID'] for p in people if p['条件']=='C3'].index('014')
others = np.unique(np.r_[group['C1'],group['C2'],np.delete(group['C3'],pos)])
anchors = np.unique(np.r_[0.000001,others,5.])
candidates = np.r_[anchors,(anchors[1:]+anchors[:-1])/2]
p_values = []
for candidate in candidates:
    replace = group['C3'].copy()
    replace[pos] = candidate
    p_values.append(omnibus({**group,'C3':replace}).pvalue)
print(f'014の順位を動かしたときのcore MTの未補正p値は {min(p_values):.4f}〜{max(p_values):.4f}。')
for r in global_tests:
    print(f"{r['指標']}　H(2) = {r['統計量']:.3f}、Holm補正後p = {r['Holm補正p']:.4f}（{r['有意性_5%']}）。")

014の順位を動かしたときのcore MTの未補正p値は 0.1222〜0.2977。
core MT　H(2) = 2.423、Holm補正後p = 0.5954（非有意）。
1/TPe　H(2) = 0.385、Holm補正後p = 0.8248（非有意）。
